In [1]:
import dotenv
import openai
import os
from pydub import AudioSegment
from glob import glob
import re
import os
import shutil
from pathlib import Path
import json

dotenv.load_dotenv()
# openai.api_key = os.environ["OPENAI_API_KEY"]
chatgpt_model_name = os.getenv('CHATGPT_MODEL')
openai.api_type = "azure"
openai.api_key = os.getenv("AZURE_OPENAI_KEY")
openai.api_base = os.getenv('AZURE_ENDPOINT')
openai.api_version = "2023-03-15-preview"

In [14]:
def f(text):
    messages=[
        {"role": "system", "content": "You are an assistant.This text contains one or two exam questions. Parse and produce a JSON array containing questions. each item should contain fields {enunciate, answer, explanation} extracted from the text. Each item has a question prompt, an answer and an explanation. Answer and explanation may be the same. Rewrite the explanation the best that you can, provided the original explanation. Enunciate is the transcription closest to original text of question prompt, fixed syntax and meaning. If there are more than one question, repeat the enunciate in each item, otherwise it is null. Output must be a JSON string parseable by python. Rewrite the text to fix syntax without changing the meaning."},
        {"role": "user", "content": text}
    ]
    return openai.ChatCompletion.create(
            engine=chatgpt_model_name,
            model="gpt-4",
            messages=messages
            )["choices"][0]["message"]["content"]

def process_file():
    with open('text/trimmed.txt', 'r') as file, open('text/processed.txt', 'w') as processed, open('text/output.json', 'w') as json_output:
        content = file.read()
        chunks = content.split('\n\n---\n\n')
        clean = []
        json_array = []

        c = 0
        for chunk in chunks:
            if chunk:
                c += 1
                print(".", end="")
                processed.write(chunk + '\n\n---\n\n')
                try:
                    clean = clean + [f(re.sub(r'[^\x20-\x7E]', '', chunk))]
                    processed.write(clean[-1])
                    try:
                        json_array = json_array + json.loads(clean[-1].replace('\\n', '\n').replace('\\\\', '\\'))
                    except:
                        processed.write('\nERROR PARSING JSON\n\n***\n\n')
                    processed.write('\n\n***\n\n')
                except:
                    processed.write("ERROR\n\n***\n\n")
                if c > 10:
                    break
        json.dump(json_array, json_output, indent=4)
        return clean

clean = process_file()

...........

In [10]:
clean

['[\n  {\n    "question": "True, False, and Explain: 60% of all agents in an economy have U=ln x + ln y, and the other 40% have U=2 ln x + ln y. All agents start with 1 unit of x and 1 unit of y.",\n    "answer": "TRUE",\n    "explanation": "Using the formula from the homework, and adjusting the ratios of agents from 50/50 to 60/40, the new ratios are calculated as follows: 0.4 * 1 * (3/1) * 40 + 0.6 * 1 * (3/2) * 40."\n  },\n  {\n    "question": "True, False, and Explain: Agents of the first type will buy approximately 0.155 units of x.",\n    "answer": "FALSE",\n    "explanation": "Each agent has a total income of 1 * 1.31 + 1 * 1 = 2.31. Agents of type 1 spend half of their income on x. Their consumption of x is equal to their total spending divided by the price: 2.31 / 2 / 1.31 = 0.882."\n  }\n]',
 '[\n  {\n    "question": "The following normal form accurately represents the \\"Big Brother/Little Brother\\" game. Player 2 Player 1 Read Play Sports Read 10,0 0,10 Play Sports 0,10 10

In [ ]:
def clean_out_path(fpath):
    out_path = os.path.join(fpath, 'out')

    # Check if the directory exists
    if os.path.exists(out_path):
        # Cleanup: delete all files in the directory
        for filename in os.listdir(out_path):
            file_path = os.path.join(out_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print('Failed to delete %s. Reason: %s' % (file_path, e))
    else:
        # Directory does not exist, so create it
        os.makedirs(out_path)

def split(audio, filename):
    ten_minutes = 20 * 60 * 1000
    pos = 0
    idx = 1
    while pos < len(audio):
        print("Saving {0}".format(filename.format(idx)))
        chunk = audio[pos:min(pos + ten_minutes, len(audio) - 1)]
        chunk.export(filename.format(idx), format="mp3")
        idx += 1
        pos += ten_minutes

for fdir in ["2017", "2018"]:
    outdir = os.path.join(os.path.curdir, fdir)
    for fname in glob("{0}/*.mp3".format(fdir)):
        print("Splitting {0}".format(fname))
        audio = AudioSegment.from_mp3(fname)
        outname = Path(os.path.splitext(fname)[0]).stem + " part {0}.mp3"
        split(audio, os.path.join(outdir, "out", outname))

In [ ]:
for fdir in ["2017", "2018"]:
    outdir = os.path.join(os.path.curdir, fdir, "out")
    n = 1
    while n > 0:
        files = glob("{0}/*.mp3".format(outdir))
        n = 0
        for fname in files:
            outname = re.sub(r"\.mp3$", ".srt", fname)
            if not os.path.exists(outname):
                with open(fname, "rb") as audio_file:
                    print("transcribing {0}".format(fname))
                    transcript = openai.Audio.transcribe("whisper-1", audio_file, response_format="srt")
                    with open(outname, "wt") as srt_file:
                        srt_file.write(transcript)
                n += 1

In [ ]:
def save_chunk(prefix, chunk, file_count):
    with open(f'{prefix}{file_count:03d}.txt', 'w') as f:
        f.write(''.join(chunk))

def process_file(filename):
    size = 0
    chunk = []
    file_count = 1
    regex = re.compile(r'^[0-9]+\.')

    with open(filename, 'rt') as f:
        for line in f:
            if regex.match(line):
                save_chunk(filename, chunk, file_count)
                chunk = []
                size = 0
                file_count += 1

            size += len(line)
            chunk.append(line)

    # Save the last chunk if it's not empty
    if chunk:
        save_chunk(filename, chunk, file_count)

for f in glob("text/*.txt"):
    process_file(f)